# Level 1 — Lesson 2: From Prompt to Program

## Coding with LLMs for Medical Imaging

**Teaching time:** ~40–45 minutes  
**Audience:** No AI experience required. Basic familiarity with running Python cells in Google Colab is helpful.

**Core question:**  
*How do I use an LLM as a coding collaborator without handing over the scientific decisions?*

In this lesson, we will use a public medical-imaging task as a running example:

> **Load 3D CT data from the Medical Segmentation Decathlon Task09 Spleen dataset and prepare one case for a future segmentation model.**

We will not train a model yet. The goal is to learn a reliable workflow for working with an LLM:

```text
vague idea
    ↓
clear specification
    ↓
implementation plan
    ↓
small code step
    ↓
run it
    ↓
inspect evidence
    ↓
revise
```

By the end of the lesson, you should be able to:

- recognize why vague coding prompts lead to unpredictable code;
- specify a coding task with useful constraints;
- ask an LLM for a plan before asking for code;
- request small, testable code changes instead of a complete project;
- ask the LLM to state assumptions rather than silently inventing them;
- verify code using shapes, values, plots, and expected behaviour;


# 0. How to use this notebook

This notebook is designed to be used **together with an LLM chat**. If you run this in Google colab, you could for example use the integrated version of Gemini. To start it, press the 'Toggle Gemini' at the bottom of the page.

Keep both open:

```text
LLM chat                     Google Colab
   │                              │
ask for one step  ───────────────→│
   │                              │
   │                         run the code
   │                              │
   │←──────────── inspect result ─┘
   │
revise the request
```

You do not need to use a particular commercial model or provider.

For this lesson, we use only **public data and generic code**. Just as a reminder:

> **Never upload patient data, clinical screenshots, internal documents, credentials, or confidential institutional information to an AI service unless that use is explicitly approved by your institution.**



# 1. Start with a deliberately bad prompt

Imagine that your goal is spleen segmentation from CT.

You could ask an LLM:

> **"Write Python code to train an AI model to segment an organ on CT."**

Before sending anything else, stop.

### Exercise

What important information is missing?

Write down at least five things the LLM would have to **guess**.

Some examples:

```text
Which organ?
Which dataset?
2D or 3D?
Which programming language?
Which AI library?
Which model?
Which loss function?
Which preprocessing?
What hardware is available?
How much training time is acceptable?
What output should the program produce?
How should we know whether it worked?
```

The problem with a vague prompt is not that the answer you get may be not as good as with a refined prompt. The issue is that:

> **The LLM has to make decisions that should be made by you.**

If those assumptions remain hidden inside generated code, it becomes difficult to fix or even understand what is happening.

# 2. Specify the task before asking for code

A useful coding request usually contains several kinds of information.

## Context

What are we working on?

```text
3D medical-image segmentation
public CT data
spleen segmentation
```

## Task

What should the code do **right now**?

```text
download the dataset
load one case
apply minimal preprocessing
display the image and label
```

## Constraints

What choices should the LLM respect?

```text
Python
Google Colab
PyTorch + MONAI
keep the code minimal
use existing library functionality
do not train a model yet
```

## Expected output

What evidence should the program produce?

```text
image shape
label shape
unique label values
one plotted CT slice with the segmentation outline
```

## Verification

How will we recognize success?

```text
image and label have aligned spatial dimensions
label contains expected discrete values
segmentation contour overlays the anatomy
```

## A much better first prompt

Copy the following prompt into your LLM.

> I am building a small educational 3D medical-image segmentation project in Google Colab.  
>   
> I want to use Python, PyTorch, and MONAI with the public Medical Segmentation Decathlon Task09 Spleen dataset.  
>   
> For now, do **not** train a neural network. I only want to:
> 1. install/import the required packages;
> 2. download the dataset using MONAI;
> 3. load one CT and its segmentation;
> 4. apply minimal preprocessing appropriate for CT and a discrete segmentation mask;
> 5. print image shape, label shape, intensity range, and unique label values;
> 6. plot one CT slice with the segmentation contour overlaid.
>   
> Keep the implementation small and readable. Prefer standard MONAI functionality instead of writing custom utilities. Do not create helper functions unless they remove meaningful duplication.
>   
> **Before writing any code, list your assumptions and propose a short implementation plan.**

Do **not** ask for the complete code yet.

# 3. Ask for a plan before asking for code

Why request a plan first?

Because a plan is much easier to review than 150 lines of code.

A reasonable answer should contain something like:

```text
1. Install MONAI and nibabel.
2. Download Task09 Spleen.
3. Select one case.
4. Load image + label.
5. Ensure channel-first representation.
6. Standardize orientation / spacing if needed.
7. Scale the CT intensity range.
8. Inspect shapes and values.
9. Plot a representative slice.
```

The exact plan may differ.

### Review the plan

Ask yourself:

- Did it add things I never requested?
- Did it silently choose important parameters?
- Can I explain why each step exists?

## If the LLM over-engineers the plan

You may receive something like:

```text
Create RandomTransforms
Create VisualizationManager
Create Logger
Create ModelFactory
...
```

None of these abstractions are bad, but for now they are unnecessary.

Try this follow-up:

> **Simplify the plan. This is a short educational notebook, not a reusable software package. Use MONAI's existing classes directly and introduce an abstraction only if it solves a concrete problem in the current notebook.**

An important skill is not to ask the LLM for more code. You should also know how to specify and to ask for *less*.

# 4. Build incrementally

Now ask the LLM for only the first implementation step:

> Implement only the environment setup and dataset download from the plan. Do not write preprocessing, visualization, or model code yet. Keep it minimal.

Run what it gives you. If it works, continue.

If it fails, do not ask:

> "Help! Fix the code!"

Instead, inspect the error first and try to understand it as best as you can.

```text
1. Read the traceback.
2. Identify the failing line.
3. Inspect the relevant variable.
4. Form a hypothesis.
5. Ask the LLM about that specific failure.
6. Make the smallest change.
7. Run again.
```
When posting error messages, be sure to not post any confidential information along with them.

We will come back to debugging in Lesson 4.

# 5. Reference setup

If your LLM-generated setup works, you can keep it.

If you get stuck, use the compact reference implementation below.

The reference is **not the only correct solution**.

In [ ]:
!pip -q install "monai==1.6.1" nibabel

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

import monai
from monai.apps import DecathlonDataset
from monai.data import Dataset
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    EnsureTyped,
)

print("MONAI:", monai.__version__)
print("PyTorch:", torch.__version__)

## Download the real public dataset

This may take a minute or two.


In [ ]:
DATA_DIR = Path("/content/msd")

raw_dataset = DecathlonDataset(
    root_dir=DATA_DIR,
    task="Task09_Spleen",
    section="training",
    transform=None,
    download=True,
    cache_rate=0.0,
)

print("Available training cases:", len(raw_dataset.data))
print("First case:")
print(raw_dataset.data[0])

# 6. Ask for one more step

Now ask your LLM:

> We have successfully downloaded the dataset.  
>   
> Write only the preprocessing needed to load one CT and its binary label. Use MONAI dictionary transforms.  
>   
> Requirements:
> - load both image and label;
> - make the channel dimension explicit;
> - use a consistent orientation;
> - resample to a fixed spacing;
> - use interpolating intensity resampling for CT and nearest-neighbour for the discrete label;
> - scale a soft-tissue CT range to [0, 1];
> - return PyTorch-compatible tensors.
>   
> Explain each transform briefly before showing the code. Do not add augmentation, patch sampling, caching, a model, or a training loop.

Again, the point is to constrain the task. The LLM certainly can generate a full working codebase for a task like spleen segmentation. We are deliberately preventing the LLM from racing ahead, because we would like to to be able to keep up i terms of understanding what it does.

# 7. Reference preprocessing

Compare your generated pipeline with this compact reference.

Try to analyse them:

> **Do they make the same scientific and computational assumptions?**

Are there any obvious differences?

In [ ]:
transforms = Compose([
    LoadImaged(keys=["image", "label"]),

    EnsureChannelFirstd(
        keys=["image", "label"]
    ),

    Orientationd(
        keys=["image", "label"],
        axcodes="RAS",
    ),

    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),
    ),

    ScaleIntensityRanged(
        keys=["image"],
        a_min=-57,
        a_max=164,
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),

    EnsureTyped(
        keys=["image", "label"]
    ),
])

Notice something important:

```python
mode=("bilinear", "nearest")
```

The same geometric operation — resampling — is applied to two different data objects.

But the interpolation differs because the **meaning of the data differs**:

```text
CT intensity      → continuous-valued
segmentation mask → discrete labels
```

An LLM may generate code that runs while still getting this wrong.

> **Executable code is not automatically scientifically correct code.**

In [ ]:
one_case = [raw_dataset.data[0]]

dataset = Dataset(
    data=one_case,
    transform=transforms,
)

sample = dataset[0]

image = sample["image"]
label = sample["label"]

print("Image shape:", tuple(image.shape))
print("Label shape:", tuple(label.shape))

print(
    "Image range:",
    float(image.min()),
    "to",
    float(image.max()),
)

print(
    "Unique label values:",
    torch.unique(label).tolist(),
)

# 8. Verification before moving on

Before asking the LLM for more code, inspect what we have.

We expect:

```text
image shape:
[1, X, Y, Z]

label shape:
[1, X, Y, Z]

image range:
approximately [0, 1]

label values:
0 and 1
```

### Ask yourself

If the code printed:

```text
label values:
[0.0, 0.13, 0.48, 0.81, 1.0]
```

would that be evidence that the program worked?

Yes, computationally it did "work".

But, no, it did not "work" in the sense of being scientifically correct.

The code executed correctly but the label was interpolated incorrectly.

# 9. Visualization is also a test

For imaging code, plotting the result is not decoration.

It is another verification step.

A tensor may have the expected shape and dtype while still being:

- flipped unexpectedly;
- misaligned with the label;
- cropped incorrectly;
- intensity-scaled badly;
- spatially corrupted.

Let's inspect one representative slice.

In [ ]:
image_np = image[0].cpu().numpy()
label_np = label[0].cpu().numpy()

slice_areas = label_np.sum(axis=(0, 1))
z = int(np.argmax(slice_areas))

plt.figure(figsize=(6, 6))

plt.imshow(
    image_np[:, :, z].T,
    cmap="gray",
    origin="lower",
)

plt.contour(
    label_np[:, :, z].T,
    levels=[0.5],
)

plt.title(f"Task09 Spleen — slice {z}")
plt.axis("off")
plt.show()

### Exercise

Before continuing, inspect the figure.

- Does the contour appear aligned with the anatomy?
- Is the CT displayed with plausible contrast?
- Did we choose a slice that actually contains the target?
- Is anything obviously rotated, stretched, or displaced?

Then ask your LLM:

> **What checks would you add to verify that this preprocessing pipeline is behaving correctly? Do not change the code yet. Give me a short list of tests or assertions and explain what failure each would detect.**

This is a much better use of an LLM than simply asking it to produce more implementation.

# 10. Ask the LLM to critique its own code

Now take the code your LLM generated during this lesson and ask:

> Review this code with a strong bias toward simplicity.  
>   
> Identify:
> - helper functions that do not add meaningful abstraction;
> - duplicated operations;
> - configuration that is unnecessary for this small task;
> - comments that merely restate the code;
> - functionality that MONAI already provides;
> - scientific assumptions that are hidden rather than explicit.
>   
> Do **not** rewrite the program yet. First explain what should be removed, retained, or clarified.

This changes the role of the LLM:

```text
first:
code generator

now:
code reviewer
```

Both roles are useful.

## A simple example of "LLM fluff"

Suppose the model generates:

```python
def rotate_image_90_degrees(image):
    rotated_image = np.rot90(image)
    return rotated_image
```

Compare with:

```python
np.rot90(image)
```

The function name is longer than the operation it hides.

That does not make the first version *incorrect*.

It makes it harder to justify.

A useful question is:

> **What meaningful concept does this abstraction add?**

If the answer is "none," deletion may improve the code.

# 11. What we deliberately did NOT ask for

During this lesson we did not ask the LLM to generate:

```text
a complete training framework
configuration files
logging infrastructure
custom Dataset classes
experiment tracking
a model zoo
command-line interfaces
deployment code
```

Not because these things are always bad.

Because we do not need them **yet**.

A reliable workflow grows complexity when the problem demands it:

```text
small verified program
        ↓
next requirement
        ↓
small verified change
        ↓
next requirement
        ↓
...
```

rather than:

```text
large imagined future project
        ↓
hundreds of unverified lines
```

# 12. A reusable prompting pattern

For future coding tasks, you can start with:

> **Context**  
> Here is the scientific/computational problem I am working on.
>
> **Task**  
> Here is the one thing I want to implement now.
>
> **Constraints**  
> Use these libraries, environment, data assumptions, and simplicity requirements.
>
> **Expected output**  
> The program should produce these concrete outputs.
>
> **Verification**  
> Add or suggest checks that would demonstrate the code behaves as intended.
>
> **Process**  
> State assumptions and propose a short plan before writing code.

This is not a magic prompt formula.

Its purpose is to make **your decisions explicit** so that the LLM has fewer important things to guess.

# 13. Exit exercise

In the next lesson, we want to continue toward an actual spleen-segmentation model on the [Medical Decathlon](http://medicaldecathlon.com/) data.

Think about the  prompt yourself.

Your request should specify only the first logical step.

### Check:

```text
Did I define the task?
Did I constrain the scope?
Did I state important assumptions?
Did I ask for evidence that the code works?
Can I understand what success should look like?
```

# Take-home message

The most useful way to work with an LLM is not:

```text
I describe an idea
        ↓
the AI writes my project
```

It is:

```text
I define the problem
        ↓
the LLM proposes
        ↓
I inspect
        ↓
we implement one step
        ↓
the computer provides evidence
        ↓
I decide what happens next
```

> **The LLM can generate code. You remain responsible for the problem definition, the scientific assumptions, and the evidence that the program is correct.**

---

## Bridge to the next lessons

So far we have only looked at small generic examples.

In the next lesson, we will design an actual medical segmentation task purely from prompting.

We will:
- use public Medical Decathlon data
- achieve a meaningful target (spleen segmentation)
- learn how to build and validate a medical AI pipeline
